In [1]:
import numpy as np

In [2]:

# ------------------------------------------
# Call the function (we'll enable prints by
# temporarily adding them to the source)
# ------------------------------------------

# For exploration, I'll create a temporary copy of the function
# with print statements enabled inside.

def get_labels_resorting_array_with_prints(
    types: np.ndarray,
    shapes: np.ndarray,
    shapes_inv: np.ndarray = None,
    transpose_neg: bool = False,
) -> np.ndarray:
    print("Starting get_labels_resorting_array_with_prints...")
    print(f"Input types: {types}")
    print(f"Input shapes:\n{shapes}")
    if shapes_inv is not None:
        print(f"Input shapes_inv:\n{shapes_inv}")
    print(f"Transpose negative: {transpose_neg}")
    n_entries = types.shape[0]
    n_types = shapes.shape[1]
    if shapes_inv is None:
        shapes_inv = shapes
    sizes = np.zeros(n_types, dtype=np.int64)
    sizes_inv = np.zeros(n_types, dtype=np.int64)
    for typ in range(n_types):
        sizes[typ] = shapes[0, typ] * shapes[1, typ]
        sizes_inv[typ] = shapes_inv[0, typ] * shapes_inv[1, typ]

    # Count the number of entries of each type
    type_nlabels = np.zeros(n_types, dtype=np.int64)
    for i_edge in range(n_entries):
        if types[i_edge] > 0:
            typ = types[i_edge]
            type_nlabels[typ] += sizes[typ] # in the ex: 4 of type 1 (size 5) 4x5 - 20.
        else:
            typ = abs(types[i_edge])
            type_nlabels[typ] += sizes_inv[typ] # in the ex: 4 of type 1 (size 5) 4x5 - 20.
    offset = np.zeros(n_types, dtype=np.int64)
    for typ in range(1, n_types):
        offset[typ] = offset[typ - 1] + type_nlabels[typ - 1]

    total_len = offset[n_types - 1] + type_nlabels[n_types - 1]
    indices = np.empty(total_len, dtype=np.int64)

    type_i = np.zeros_like(sizes)
    i = 0

    print("=" * 60)
    print("Initial state:")
    print(f"  n_entries = {n_entries}, n_types = {n_types}")
    print(f"  sizes      = {sizes}")
    print(f"  sizes_inv  = {sizes_inv}")
    print(f"  type_nlabels = {type_nlabels}")
    print(f"  offset     = {offset}")
    print(f"  total_len  = {total_len}")
    print("=" * 60)

    for i_edge in range(n_entries):
        typ = types[i_edge]
        abs_type = abs(typ)
        block_size = sizes[abs_type]
        start = offset[abs_type] + type_i[abs_type]

        print(f"\n--- Edge {i_edge}: type={typ}, abs_type={abs_type}, "
              f"block_size={block_size}, start={start} ---")

        if transpose_neg and typ < 0:
            cols, rows = shapes[0, abs_type], shapes[1, abs_type]
            print(f"  Transposing: original shape ({shapes[0, abs_type]}x{shapes[1, abs_type]}) "
                  f"=> new dims ({rows}x{cols})")
            for jrow in range(rows):
                for jcol in range(cols):
                    idx_val = start + jcol * rows + jrow
                    indices[i] = idx_val
                    print(f"    indices[{i}] = {idx_val}  (jrow={jrow}, jcol={jcol})")
                    i += 1
        elif typ < 0:
            block_size = sizes_inv[abs_type]
            print(f"  Negative type (no transpose): filling indices[{i} : {i+block_size}] "
                  f"with {start} ... {start+block_size-1}")
            for j in range(start, start + block_size):
                indices[i] = j
                print(f"    indices[{i}] = {j}")
                i += 1
        else:
            block_size = sizes[abs_type]
            print(f"  Normal (no transpose): filling indices[{i} : {i+block_size}] "
                  f"with {start} ... {start+block_size-1}")
            for j in range(start, start + block_size):
                print(f"    indices[{i}] = {j}")
                indices[i] = j
                i += 1

        type_i[abs_type] += block_size
        print(f"  Updated type_i[{abs_type}] = {type_i[abs_type]}")

    print("\n" + "=" * 60)
    print("Final indices:", indices)
    print("=" * 60)
    return indices

In [3]:


# Example data: 
# for the case 
# point_1 = PointBasis("A", R=2, basis="0e", basis_convention="spherical", matrix_role='row')  # "0e"
# point_2 = PointBasis("A", R=2, basis="2x0e", basis_convention="spherical", matrix_role='col')
# point_3 = PointBasis("B", R=5, basis="0e + 1o", basis_convention="spherical", matrix_role='row')
# point_4 = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical", matrix_role='col')

# types:  [ 1 -1  1 -1]
# shapes:  [[1 1 4 4]
#  [2 5 2 5]]
# transpose_neg:  False
shapes = np.array([
    [1, 1, 4, 4],   # rows per type
    [2, 5, 2, 5]    # cols per type
], dtype=np.int64)

# Edge order: type 0, type 1, then type 1 but transposed
types = np.array([1, -1, 1, -1], dtype=np.int64)




# ------------------------------------------
# Run the exploration
# ------------------------------------------
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=True
)


# # Apply to the labels to see the reordered result
# sorted_labels = labels[indices]

# print("\nOriginal type‑grouped labels (flat):")
# print(labels)
# print("\nResorting indices:")
# print(indices)
# print("\nLabels after reordering (edge‑wise order):")
# print(sorted_labels)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1]
Input shapes:
[[1 1 4 4]
 [2 5 2 5]]
Transpose negative: True
Initial state:
  n_entries = 4, n_types = 4
  sizes      = [ 2  5  8 20]
  sizes_inv  = [ 2  5  8 20]
  type_nlabels = [ 0 20  0  0]
  offset     = [ 0  0 20 20]
  total_len  = 20

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=5 ---
  Transposing: original shape (1x5) => new dims (5x1)
    indices[5] = 5  (jrow=0, jcol=0)
    indices[6] = 6  (jrow=1, jcol=0)
    indices[7] = 7  (jrow=2, jcol=0)
    indices[8] = 8  (jrow=3, jcol=0)
    indices[9] = 9  (jrow=4, jcol=0)
  Updated type_i[1] = 10

--- Edge 2: type=1, abs_type=1, block_size=5, start=10 ---
  Normal (no transpose): filling indices[10 : 15] with 10

In [4]:

print("\nResorting indices:")
print(indices)


Resorting indices:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


In [5]:
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1]
Input shapes:
[[1 1 4 4]
 [2 5 2 5]]
Transpose negative: False
Initial state:
  n_entries = 4, n_types = 4
  sizes      = [ 2  5  8 20]
  sizes_inv  = [ 2  5  8 20]
  type_nlabels = [ 0 20  0  0]
  offset     = [ 0  0 20 20]
  total_len  = 20

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=5 ---
  Negative type (no transpose): filling indices[5 : 10] with 5 ... 9
    indices[5] = 5
    indices[6] = 6
    indices[7] = 7
    indices[8] = 8
    indices[9] = 9
  Updated type_i[1] = 10

--- Edge 2: type=1, abs_type=1, block_size=5, start=10 ---
  Normal (no transpose): filling indices[10 : 15] with 10 ... 14
    indices[10] = 10
    indices[11] = 11
    indices[12] = 12
   

In [6]:
shapes_inv = np.array([
    [1, 4, 4],   # rows per type
    [2, 2, 5]    # cols per type
], dtype=np.int64)
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    shapes_inv=shapes_inv,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Input shapes_inv:
[[1 4 4]
 [2 2 5]]
Transpose negative: False
Initial state:
  n_entries = 4, n_types = 3
  sizes      = [ 2  5 20]
  sizes_inv  = [ 2  8 20]
  type_nlabels = [ 0 26  0]
  offset     = [ 0  0 26]
  total_len  = 26

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=5 ---
  Negative type (no transpose): filling indices[5 : 13] with 5 ... 12
    indices[5] = 5
    indices[6] = 6
    indices[7] = 7
    indices[8] = 8
    indices[9] = 9
    indices[10] = 10
    indices[11] = 11
    indices[12] = 12
  Updated type_i[1] = 13

--- Edge 2: type=1, abs_type=1, block_size=5, start=13 ---
  Normal (no transpose): filling indices[13 : 

In [8]:
types = np.array([1, -1, 1, -1, 1, -1, 1, -1], dtype=np.int64)
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1  1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Transpose negative: False
Initial state:
  n_entries = 8, n_types = 3
  sizes      = [ 2  5 20]
  sizes_inv  = [ 2  5 20]
  type_nlabels = [ 0 40  0]
  offset     = [ 0  0 40]
  total_len  = 40

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=5 ---
  Negative type (no transpose): filling indices[5 : 10] with 5 ... 9
    indices[5] = 5
    indices[6] = 6
    indices[7] = 7
    indices[8] = 8
    indices[9] = 9
  Updated type_i[1] = 10

--- Edge 2: type=1, abs_type=1, block_size=5, start=10 ---
  Normal (no transpose): filling indices[10 : 15] with 10 ... 14
    indices[10] = 10
    indices[11] = 11
    indices[12] = 12
    ind

In [7]:
types = np.array([1, -1, 1, -1, 1, -1, 1, -1], dtype=np.int64)
shapes_inv = np.array([
    [1, 4, 4],   # rows per type
    [2, 2, 5]    # cols per type
], dtype=np.int64)
shapes = np.array([
    [1, 1, 4],   # rows per type
    [2, 5, 5]    # cols per type
], dtype=np.int64)
indices = get_labels_resorting_array_with_prints(
    types=types,
    shapes=shapes,
    shapes_inv=shapes_inv,
    transpose_neg=False
)

Starting get_labels_resorting_array_with_prints...
Input types: [ 1 -1  1 -1  1 -1  1 -1]
Input shapes:
[[1 1 4]
 [2 5 5]]
Input shapes_inv:
[[1 4 4]
 [2 2 5]]
Transpose negative: False
Initial state:
  n_entries = 8, n_types = 3
  sizes      = [ 2  5 20]
  sizes_inv  = [ 2  8 20]
  type_nlabels = [ 0 52  0]
  offset     = [ 0  0 52]
  total_len  = 52

--- Edge 0: type=1, abs_type=1, block_size=5, start=0 ---
  Normal (no transpose): filling indices[0 : 5] with 0 ... 4
    indices[0] = 0
    indices[1] = 1
    indices[2] = 2
    indices[3] = 3
    indices[4] = 4
  Updated type_i[1] = 5

--- Edge 1: type=-1, abs_type=1, block_size=5, start=5 ---
  Negative type (no transpose): filling indices[5 : 13] with 5 ... 12
    indices[5] = 5
    indices[6] = 6
    indices[7] = 7
    indices[8] = 8
    indices[9] = 9
    indices[10] = 10
    indices[11] = 11
    indices[12] = 12
  Updated type_i[1] = 13

--- Edge 2: type=1, abs_type=1, block_size=5, start=13 ---
  Normal (no transpose): filling i